In [ ]:
import os

In [ ]:
import fitz
from PIL import Image
import io
import numpy as np
from paddleocr import PPStructureV3
from bs4 import BeautifulSoup

In [ ]:
pipeline = PPStructureV3()

In [ ]:
def html_table_to_markdown(html_text):
    """
    将HTML表格转换为Markdown格式
    """
    soup = BeautifulSoup(html_text, 'html.parser')
    table = soup.find('table')
    
    if not table:
        return html_text  # 没有表格，原样返回
    
    rows = []
    for tr in table.find_all('tr'):
        row = []
        for cell in tr.find_all(['td', 'th']):
            # 获取文本，去除多余空白
            text = cell.get_text(strip=True)
            # 处理跨列
            colspan = int(cell.get('colspan', 1))
            row.append(text)
            # 如果有跨列，补充空单元格
            for _ in range(colspan - 1):
                row.append('')
        rows.append(row)
    
    # 确定最大列数
    max_cols = max(len(row) for row in rows) if rows else 0
    
    # 补齐每行的列数
    for row in rows:
        while len(row) < max_cols:
            row.append('')
    
    # 生成Markdown
    md_lines = []
    for i, row in enumerate(rows):
        md_lines.append('| ' + ' | '.join(row) + ' |')
        # 表头后添加分隔符
        if i == 0:
            md_lines.append('| ' + ' | '.join(['---'] * max_cols) + ' |')
    
    return '\n'.join(md_lines)

In [ ]:
def parse_image(image_bytes, ocr_engine):
    image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    img_array = np.array(image)
    output = ocr_engine.predict(img_array)
    markdown_texts = ''
    for res in output:
        markdown_texts += res.markdown['markdown_texts']
    return markdown_texts

def parse_text(block):
    return block['lines'][0]['spans'][0]['text']

def parse_table(block):
    lines =  block['lines']
    markdown_texts = table_to_markdown(lines)
    return markdown_texts

def table_to_markdown(lines, y_tolerance=5, x_tolerance=20):
    """
    将PDF解析的lines转换为Markdown表格
    y_tolerance: y坐标差异小于此值视为同一行
    x_tolerance: x坐标差异小于此值视为同一列
    """
    
    # 提取有效文本行（过滤空内容）
    valid_lines = []
    for line in lines:
        text = line['spans'][0]['text']
        if text:
            x0, y0, x1, y1 = line['bbox']
            valid_lines.append({
                'text': text,
                'x0': x0,
                'y0': y0,
                'x1': x1,
                'y1': y1
            })
    
    # 按y0分组（同一行）
    rows = []
    current_row = []
    last_y = None
    
    for line in sorted(valid_lines, key=lambda x: (x['y0'], x['x0'])):
        if last_y is None or abs(line['y0'] - last_y) <= y_tolerance:
            current_row.append(line)
        # y0差异大
        else:
            if current_row:
                rows.append(sorted(current_row, key=lambda x: x['x0']))
            current_row = [line]
        last_y = line['y0']
    # 最后一行数据
    if current_row:
        rows.append(sorted(current_row, key=lambda x: x['x0']))
    
    
    # 确定列数（取最大行的列数）
    max_cols = max(len(row) for row in rows) if rows else 0
    
    md_lines = []
    
    for i, row in enumerate(rows):
        cells = [cell['text'] for cell in row]
        # 补齐列数（空列)
        while len(cells) < max_cols:
            cells.append('')
        md_lines.append('|' + '|'.join(cells) + '|')
        
        # 表头后添加分隔行
        if i == 0:
            md_lines.append('|' + '|'.join(['---'] * max_cols) + '|')
    
    return '\n'.join(md_lines)

In [ ]:
doc = fitz.open('text.pdf')

In [ ]:
for page_num in range(len(doc)):
    print('----第' + str(page_num + 1) + '页----')
    page = doc[page_num]
    blocks = page.get_text('dict')['blocks']
    for block in sorted(blocks, key=lambda x: (x["bbox"][1], x["bbox"][0])):
        if block['type'] == 0: 
            if len(block['lines']) == 1:
                texts = parse_text(block)
                print("prase_text: " , texts)
            else:
                texts = parse_table(block)
                print("prase_table: " , texts)

        if block['type'] == 1: 
            img_bytes = block['image']
            markdown_texts = parse_image(image_bytes=img_bytes, ocr_engine=pipeline)
            print("prase_image: " , markdown_texts)

block["bbox"]: [x0, y0, x1, y1]
x0 = 左边缘距离
y0 = 上边缘距离（距离页面顶部的距离）
x1 = 右边缘距离  
y1 = 下边缘距离（距离页面顶部的距离）

统一按 bbox[1] (y0, 顶部坐标) 排序，同行则按 bbox[0] (x0)：

# paddle ocr

In [ ]:
from paddleocr import PPStructureV3

In [ ]:
pipeline = PPStructureV3()
output = pipeline.predict("../text.png")


In [ ]:
output[0]

In [ ]:
markdown_texts = output[0].markdown['markdown_texts'].split('\n')
with open('../output/text.md','w',encoding='utf-8') as f:
    for markdown_text in markdown_texts:
        print(markdown_text)
        f.write(markdown_text + '\n')

In [ ]:
o = pipeline.predict('./text.pdf')

In [ ]:
for res in o:
    markdown_texts = res.markdown['markdown_texts'].split('\n')
    with open('../output/text.md','a',encoding='utf-8') as f:
        for markdown_text in markdown_texts:
            print(markdown_text)
            f.write(markdown_text + '\n')